## Camada Bronze

### Leitura e ingestão dos arquivos:
- `base_atendentes.csv`
- `base_motivos.csv`
- `canais.csv`
- `chamados.csv`
- `Chamados_Hora.CSV`    
- `clientes.csv`
- `custos.csv`
- `pesquisa_satisfacao.csv`

### Nomeclatura / Padronização:
- `ft_` para tabelas fato:

- `dm_` para tabelas dimensão

### Tabelas retornadas:

- `dm_base_atendentes`
- `dm_base_motivos`
- `dm_canais`
- `ft_chamados`
- `ft_clientes`
- `ft_custos`
- `ft_chamados_hora`
- `ft_pesquisa_satisfacao`

## Setup do Ambiente
- Importação de bibliotecas essenciais do PySpark
- Definição dos paths utilizados para as tabelas
- Criação do catálogo `catalogo`, schemas `bronze_db_name` e volume `landing_volume`

In [0]:
from pyspark.sql import functions as F

### Definição de Variáveis Globais
- Centralizamos os nomes de catálogos, bancos de dados e caminhos.
- **Boas Práticas:** Evitar "hardcoding" (escrever o caminho diretamente no código várias vezes). Se o nome do catálogo mudar no futuro, alteramos apenas aqui.

In [0]:
catalogo = "medalhao_credit"
bronze_db_name = "bronze_credit"
silver_db_name = "silver_credit"
landing_volume = "data"
volume_path = f"/Volumes/{catalogo}/default/{landing_volume}"

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalogo};")
spark.sql(f"USE CATALOG {catalogo};")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_db_name};")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_db_name};")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {landing_volume};")
spark.sql(f"USE SCHEMA {bronze_db_name};")

## Ingestão dos Arquivos / Função de Ingestão
- **Função** `ingest_csv`: responsável por ler arquivos CSV, adicionar metadados e salvar como tabela Delta.
- **Coluna `data_ingestao`:** adicionada como metadado para fins de auditoria, permitindo rastrear quando os dados foram inseridos no datalake.
- **Formato Delta:** Salvamos como tabela Delta para garantir transações ACID e performance.
- **`inferSchema=True`:** Na camada Bronze, aceitamos a inferência automática para agilizar a ingestão. A tipagem forte será forçada na camada Silver.

In [0]:
def ingest_csv(nome_arquivo:str, nome_tabela:str, tem_header=False):
    caminho_completo = f"{volume_path}/{nome_arquivo}"
    try:
        df = spark.read.csv(caminho_completo, header=tem_header, inferSchema=True)

        if df.count() == 0:
            raise ValueError(f'O arquivo {nome_arquivo} está vazio ou não pôde ser lido.')

        df_with_metadata = df.withColumn('data_ingestao', F.current_timestamp())

        df_with_metadata.write \
            .format('delta') \
            .mode('overwrite') \
            .option('inferSchema', 'true') \
            .saveAsTable(f'{catalogo}.{bronze_db_name}.{nome_tabela}')

        print(f'Tabela bronze_credit.{nome_tabela} criada com sucesso!\n')

    except Exception as e:
        print(f'Erro ao processar {nome_tabela}: {str(e)}')

In [0]:
nomes_arquivos = [
    "base_atendentes.csv",
    "base_motivos.csv",
    "canais.csv",
    "chamados.csv",
    "clientes.csv",
    "custos.csv",
    "pesquisa_satisfacao.csv"
]

nomes_tabelas = [
    "dm_base_atendentes",
    "dm_base_motivos",
    "dm_canais",
    "ft_chamados",
    "ft_clientes",
    "ft_custos",
    "ft_pesquisa_satisfacao"
]


for i in range(0, len(nomes_arquivos)):
    ingest_csv(nomes_arquivos[i], nomes_tabelas[i])

ingest_csv('Chamados_Hora.CSV', 'ft_chamados_hora', True)

## Visualização das Tabelas
- Leitura das tabelas criadas para checagem do funcionamento da ingestão
- Exibimos os valores únicos de cada tabela, caso haja menos de 10 distintos.

In [0]:
def view_tables(tabelas, limite=10):
    for tabela in tabelas:
        print(f"Tabela: {tabela}")
        df = spark.table(f"{catalogo}.{bronze_db_name}.{tabela}")
        for col in df.columns:
            if col == 'data_ingestao':
                continue
            unique_values = [row[col] for row in df.select(col).distinct().collect()]
            if len(unique_values) < 10:
                print(f"Valores únicos da coluna '{col}': {unique_values}")
        display(df.limit(limite))

view_tables(nomes_tabelas)